# Detecting Underserved Transit Zones — Supply-Gap Clustering

We rank San Diego census tracts by a **transit supply gap** — how far transit *demand* outstrips transit *supply* — then group the worst tracts into contiguous geographic **zones** a planner could act on.

**Framing: supply gap.** For each tract we build a `gap_score = z(need) - z(supply)`. A high score means the tract has high demand for transit (dense, lots of renters, lower income) but few transit stops per km². Tracts in the top quartile of gap score are flagged *underserved*.

**The clustering step.** Underserved tracts are scattered across the map. We use **single-linkage clustering on tract centroids with a distance cutoff** to merge tracts whose centroids fall within `ZONE_EPS_M` metres of each other into one contiguous zone. This is equivalent to DBSCAN with `min_samples=1`: any chain of nearby underserved tracts becomes a single zone, and isolated tracts stay as singletons.

**Inputs**
- `data/san-diego-census-geopandas/data/San Diego.csv` &mdash; one row per tract with demographics + WKT `geometry`.
- `output_csvs/san_diego_transit_stop_density.csv` &mdash; one row per tract with `stops_per_km2`.

**Output**
- `output_csvs/san_diego_underserved_zones.csv` &mdash; every tract with its `gap_score`, `underserved` flag, and `zone_id`.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely import wkt
from scipy.cluster.hierarchy import linkage, fcluster

import warnings
warnings.simplefilter(action='ignore', category=Warning)
pd.set_option('display.max_columns', None)

# --- tunable parameters ---
UNDERSERVED_Q = 0.75          # tracts at/above this gap-score quantile are flagged underserved
ZONE_EPS_M    = 1200          # max centroid distance (m) to merge adjacent tracts into one zone
METERS_CRS    = 'EPSG:32611'  # UTM 11N — accurate distances/areas for San Diego

## 1. Load and join the tracts

Both tables are keyed by `GEOID`. The census table carries the `geometry` (WKT, stored in Web Mercator / EPSG:3857), which we parse into a `GeoDataFrame` for the spatial steps later.

In [ ]:
census  = pd.read_csv('data/san-diego-census-geopandas/data/San Diego.csv')
density = pd.read_csv('output_csvs/san_diego_transit_stop_density.csv')

df = census.merge(
    density[['GEOID', 'num_stops', 'area_km2', 'stops_per_km2']],
    on='GEOID', how='inner',
)

gdf = gpd.GeoDataFrame(df, geometry=df['geometry'].apply(wkt.loads), crs='EPSG:3857')
print(f'{len(census):,} census tracts × {len(density):,} density tracts → {len(gdf):,} matched')
gdf[['GEOID', 'NAME', 'total_pop', 'median_hh_income', 'stops_per_km2']].head()

## 2. Build the demand (need) and supply features

**Need** is a blend of three standardized proxies, each pointing the same way (higher = more transit need):
- `pop_density` &mdash; people per km² (more people, more demand).
- `renter_share` &mdash; rented ÷ total housing units (renters are less car-dependent).
- **inverted** `median_hh_income` &mdash; lower income tracts rely on transit more, so we use `z(-income)`.

We z-score each proxy (so dollars don't dominate ratios) and average them into a single `need_index`.

In [ ]:
gdf['pop_density']  = gdf['total_pop'] / gdf['area_km2']
gdf['renter_share'] = gdf['total_rented'] / gdf['total_housing_units']

need_cols = ['pop_density', 'renter_share', 'median_hh_income']
gdf = gdf.dropna(subset=need_cols + ['stops_per_km2']).copy()

def z(s):
    s = s.astype(float)
    return (s - s.mean()) / s.std(ddof=0)

need = pd.DataFrame({
    'z_pop_density':  z(gdf['pop_density']),
    'z_renter_share': z(gdf['renter_share']),
    'z_low_income':   z(-gdf['median_hh_income']),   # invert: low income -> high need
})
gdf['need_index'] = need.mean(axis=1)
need.describe()

## 3. The supply-gap score

`stops_per_km2` is right-skewed, so we log-compress it before standardizing. The gap is then simply standardized need minus standardized supply:

$$\text{gap\_score} = z(\text{need\_index}) - z(\log(1 + \text{stops\_per\_km}^2))$$

Positive → demand outstrips supply (underserved); negative → well served relative to demand.

In [ ]:
gdf['z_need']   = z(gdf['need_index'])
gdf['z_supply'] = z(np.log1p(gdf['stops_per_km2']))
gdf['gap_score'] = gdf['z_need'] - gdf['z_supply']

(gdf[['GEOID', 'NAME', 'need_index', 'stops_per_km2', 'gap_score']]
    .sort_values('gap_score', ascending=False)
    .head(10))

## 4. Flag the underserved tracts

Tracts at or above the `UNDERSERVED_Q` quantile of `gap_score` are flagged. With the default `0.75` that's the worst-served quarter of tracts.

In [ ]:
threshold = gdf['gap_score'].quantile(UNDERSERVED_Q)
gdf['underserved'] = gdf['gap_score'] >= threshold

n = int(gdf['underserved'].sum())
print(f'gap-score threshold (q={UNDERSERVED_Q}): {threshold:.3f}')
print(f'{n} of {len(gdf)} tracts flagged underserved ({n / len(gdf):.0%})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(gdf['gap_score'], bins=40, color='#4c78a8')
axes[0].axvline(threshold, color='#e4572e', lw=2, label=f'threshold = {threshold:.2f}')
axes[0].set(title='Supply-gap score distribution',
            xlabel='gap_score  (z_need − z_supply)', ylabel='# tracts')
axes[0].legend()

colors = np.where(gdf['underserved'], '#e4572e', '#bbbbbb')
axes[1].scatter(gdf['z_supply'], gdf['z_need'], c=colors, s=16, alpha=0.7, edgecolor='none')
axes[1].set(title='Need vs supply  (red = underserved)',
            xlabel='z_supply  (transit stops)', ylabel='z_need  (demand)')
plt.tight_layout(); plt.show()

## 5. Cluster underserved tracts into contiguous zones

We take only the flagged tracts, compute their centroids in projected metres, and run **single-linkage** hierarchical clustering with a distance cutoff of `ZONE_EPS_M`. `fcluster(..., criterion='distance')` cuts the tree so that any two tracts within the cutoff (directly or via a chain of neighbours) land in the same zone — the connected-components behaviour of DBSCAN with `min_samples=1`, with no extra dependency.

_(Swap in `sklearn.cluster.DBSCAN(eps=ZONE_EPS_M, min_samples=2)` here if you'd rather drop isolated tracts as noise.)_

In [ ]:
under = gdf[gdf['underserved']].copy()

cent = under.geometry.to_crs(METERS_CRS).centroid
coords = np.c_[cent.x.to_numpy(), cent.y.to_numpy()]

if len(coords) > 1:
    Z = linkage(coords, method='single')
    zone_ids = fcluster(Z, t=ZONE_EPS_M, criterion='distance')
else:
    zone_ids = np.ones(len(coords), dtype=int)

under['zone_id'] = zone_ids
sizes = under['zone_id'].value_counts()
under['zone_size'] = under['zone_id'].map(sizes)

print(f'{under["zone_id"].nunique()} zones from {len(under)} underserved tracts (eps={ZONE_EPS_M} m)')
print(f'multi-tract zones: {(sizes >= 2).sum()};  singleton tracts: {(sizes == 1).sum()}')

## 6. Zone summary

The biggest, worst-gap zones are the priorities. Each row aggregates the tracts in a zone.

In [ ]:
zone_summary = (under.groupby('zone_id')
    .agg(n_tracts     = ('GEOID', 'size'),
         total_pop    = ('total_pop', 'sum'),
         mean_income  = ('median_hh_income', 'mean'),
         mean_gap     = ('gap_score', 'mean'),
         total_stops  = ('num_stops', 'sum'),
         mean_density = ('stops_per_km2', 'mean'))
    .sort_values(['n_tracts', 'mean_gap'], ascending=False))
zone_summary.head(15)

## 7. Map the zones

All tracts in light grey; each contiguous multi-tract underserved zone gets its own colour, with isolated tracts in mid-grey. The side key lists every zone's **mean gap score**, ordered most → least underserved.

In [ ]:
import matplotlib.patches as mpatches

under['zone_gap'] = under.groupby('zone_id')['gap_score'].transform('mean')
multi = under[under['zone_size'] >= 2]
solo  = under[under['zone_size'] == 1]

# order multi-tract zones most -> least underserved, give each a fixed colour
zone_order = multi.groupby('zone_id')['zone_gap'].first().sort_values(ascending=False)
cmap = plt.get_cmap('tab20')
zone_color = {zid: cmap(i % 20) for i, zid in enumerate(zone_order.index)}

fig, ax = plt.subplots(figsize=(11, 11))
gdf.to_crs(METERS_CRS).plot(ax=ax, color='#eeeeee', edgecolor='white', linewidth=0.3)

multi_m = multi.to_crs(METERS_CRS)
for zid in zone_order.index:
    multi_m[multi_m['zone_id'] == zid].plot(ax=ax, color=zone_color[zid],
                                            edgecolor='black', linewidth=0.3)
solo.to_crs(METERS_CRS).plot(ax=ax, color='#999999', edgecolor='black', linewidth=0.3)

handles = [mpatches.Patch(color=zone_color[zid], label=f'zone {zid}:  {gap:.2f}')
           for zid, gap in zone_order.items()]
handles.append(mpatches.Patch(color='#999999', label='isolated tracts'))
ax.legend(handles=handles, title='zone mean gap score\n(most → least underserved)',
          loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=8, frameon=False)

ax.set_title(f'Underserved transit zones (top {1 - UNDERSERVED_Q:.0%} supply gap)\n'
             f'coloured = contiguous multi-tract zones · grey = isolated tracts')
ax.set_axis_off()
plt.tight_layout(); plt.show()

## 8. Write the output

One row per tract with its gap score, flag, and zone assignment (`zone_id` is null for tracts that aren't underserved).

In [ ]:
out = (gdf[['GEOID', 'NAME', 'total_pop', 'median_hh_income', 'pop_density',
            'renter_share', 'stops_per_km2', 'need_index', 'gap_score', 'underserved']]
       .merge(under[['GEOID', 'zone_id', 'zone_size']], on='GEOID', how='left'))
out['zone_id'] = out['zone_id'].astype('Int64')

path = 'output_csvs/san_diego_underserved_zones.csv'
out.sort_values('gap_score', ascending=False).to_csv(path, index=False)
print(f'wrote {path}  ({len(out)} tracts, {int(out["underserved"].sum())} underserved)')
out.sort_values('gap_score', ascending=False).head(10)